# Μάθημα 05 - Πρακτική RAG


## Setup

Αυτό το σημειωματάριο παρουσιάζει το πρότυπο Agentic RAG (Retrieval-Augmented Generation) χρησιμοποιώντας το Microsoft Agent Framework.

**Προαπαιτούμενα:**
- `AZURE_SEARCH_SERVICE_ENDPOINT` — το endpoint της υπηρεσίας Azure AI Search σας
- `AZURE_SEARCH_API_KEY` — το κλειδί API της υπηρεσίας Azure AI Search σας
- Ανάθεση Azure OpenAI ρυθμισμένη μέσω μεταβλητών περιβάλλοντος
- Πιστοποιημένος Azure CLI (`az login`)


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Azure AI Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Τι είναι το Agentic RAG;

Το παραδοσιακό RAG ακολουθεί ένα σταθερό pipeline: ανάκτηση εγγράφων, στη συνέχεια δημιουργία απάντησης. Το **Agentic RAG** προχωράει παραπάνω δίνοντας στον πράκτορα αυτονομία να αποφασίσει **πότε** και **πώς** θα ανακτήσει πληροφορίες.

Με το Agentic RAG, ο πράκτορας μπορεί να:
- **Αποφασίζει** αν χρειάζεται ανάκτηση πριν απαντήσει σε μια ερώτηση
- **Επιλέγει** ποια πηγή δεδομένων ή εργαλείο θα ρωτήσει
- **Αξιολογεί** τα αποτελέσματα ανάκτησης και πραγματοποιεί επανειλημμένες αναζητήσεις αν η πρώτη προσπάθεια δεν είναι επαρκής
- **Συνδυάζει** πληροφορίες από πολλαπλά βήματα ανάκτησης σε μια συνεκτική απάντηση

Αυτό κάνει τον πράκτορα πιο ευέλικτο και ακριβή σε σύγκριση με ένα στατικό pipeline ανάκτησης-μετά-δημιουργίας.


## Δημιουργία Εργαλείου Αναζήτησης

Στο Agentic RAG, οι εξωτερικές πηγές δεδομένων είναι ενσωματωμένες ως **εργαλεία** που ο πράκτορας μπορεί να καλεί κατόπιν ζήτησης. Αυτό επιτρέπει στον πράκτορα να αντιμετωπίζει την ανάκτηση ως απλή ενέργεια που μπορεί να εκτελέσει, αντί για υποχρεωτικό βήμα.

Παρακάτω ορίζουμε μια βάση γνώσεων για ταξίδια και την εκθέτουμε ως εργαλείο που ο πράκτορας μπορεί να καλέσει για να αναζητήσει πληροφορίες προορισμού.


In [ ]:
TRAVEL_KNOWLEDGE_BASE = {
    "Barcelona": "Barcelona is Spain's cosmopolitan capital of Catalonia. Best visited Mar-May or Sep-Nov. Known for Gaudí architecture, La Rambla, beaches. Average daily cost: $150-200.",
    "Tokyo": "Tokyo is Japan's capital, mixing ultramodern with traditional. Best visited Mar-Apr (cherry blossoms) or Oct-Nov. Known for Shibuya, temples, sushi. Average daily cost: $200-250.",
    "Paris": "Paris is France's capital and a global center for art, fashion, and culture. Best visited Apr-Jun or Sep-Oct. Known for Eiffel Tower, Louvre, cuisine. Average daily cost: $180-250.",
    "Cape Town": "Cape Town sits on South Africa's southwest tip. Best visited Nov-Mar. Known for Table Mountain, wine regions, wildlife. Average daily cost: $100-150.",
}


@tool(approval_mode="never_require")
def search_travel_knowledge(
    query: Annotated[str, "The search query about a travel destination"]
) -> str:
    """Search the travel knowledge base for destination information."""
    results = []
    for destination, info in TRAVEL_KNOWLEDGE_BASE.items():
        if query.lower() in destination.lower() or any(
            word in info.lower() for word in query.lower().split()
        ):
            results.append(f"**{destination}**: {info}")
    return (
        "\n\n".join(results)
        if results
        else "No matching destinations found in the knowledge base."
    )

## Δημιουργία του Πράκτορα RAG

Τώρα δημιουργούμε έναν πράκτορα που έχει εντολή να **ανακτά πάντα πληροφορίες πριν απαντήσει**. Ο πράκτορας χρησιμοποιεί το εργαλείο `search_travel_knowledge` για να στηρίξει τις απαντήσεις του στη βάση γνώσεων αντί να βασίζεται στα δικά του δεδομένα εκπαίδευσης.


In [ ]:
agent = client.as_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGAgent",
    instructions="""You are a knowledgeable travel advisor. Before answering questions about destinations:
1. ALWAYS search the travel knowledge base first
2. Base your answers on retrieved information
3. If information is not in the knowledge base, say so clearly
4. Provide specific details like costs, best seasons, and highlights.""",
)

response = await agent.run(
    "I'm interested in visiting somewhere with great architecture. What destinations would you recommend?",
    )
print(response)

## Επαναληπτική Ανάκτηση — Το Πρότυπο Δημιουργός-Ελεγκτής

Ένα βασικό πλεονέκτημα του Agentic RAG είναι η **επαναληπτική ανάκτηση**. Ο πράκτορας μπορεί να εκτελέσει πολλούς γύρους αναζήτησης για να επαληθεύσει, να βελτιώσει ή να επεκτείνει τα αρχικά του ευρήματα — παρόμοια με μια ροή εργασίας "δημιουργός-ελεγκτής":

1. **Βήμα Δημιουργού**: Ο πράκτορας ανακτά αρχικές πληροφορίες και συντάσσει μια απάντηση.
2. **Βήμα Ελεγκτή**: Ο πράκτορας εκτελεί πρόσθετες ανακτήσεις για να επαληθεύσει λεπτομέρειες ή να καλύψει κενά.

Παρακάτω, ο πράκτορας λαμβάνει μια ερώτηση που απαιτεί σύγκριση πολλαπλών προορισμών, προκαλώντας τον να αναζητήσει αρκετές φορές.


In [ ]:
checker_agent = client.as_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGCheckerAgent",
    instructions="""You are a meticulous travel advisor who double-checks recommendations.
When answering travel questions:
1. Search for relevant destinations first
2. For each destination found, search again with the destination name to get full details
3. Compare the options using verified information
4. Present a final recommendation with specific costs, best travel times, and highlights
5. If any detail seems incomplete, search once more to confirm before responding.""",
)

response = await checker_agent.run(
    "I have a $175/day budget and want to travel in April. Which destinations fit my budget and timing?",
    )
print(response)

## Περίληψη

Σε αυτό το μάθημα μάθατε πώς να δημιουργήσετε ένα σύστημα **Agentic RAG** χρησιμοποιώντας το Microsoft Agent Framework:

- Το **Agentic RAG** επιτρέπει στους παράγοντες να αποφασίζουν αυτόνομα πότε να ανακτούν πληροφορίες, κάνοντας την ανάκτηση δυναμική αντί για σταθερή.
- **Εργαλεία ως πηγές δεδομένων**: Εξωτερικές βάσεις γνώσεων (όπως το Azure AI Search) συσκευάζονται ως εργαλεία που μπορεί να καλεί ο παράγοντας.
- **Επαναληπτική ανάκτηση**: Το μοτίβο maker-checker επιτρέπει στον παράγοντα να εκτελεί πολλούς γύρους ανάκτησης — αναζητώντας, επαληθεύοντας και βελτιώνοντας — πριν παράγει τελικό αποτέλεσμα.

Σε παραγωγή, θα αντικαθιστούσατε τη βάση `TRAVEL_KNOWLEDGE_BASE` στη μνήμη με έναν πραγματικό δείκτη Azure AI Search για να διαχειριστείτε μεγάλου όγκου ανάκτηση ταξιδιωτικών εγγράφων.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Αποποίηση ευθυνών**:
Αυτό το έγγραφο έχει μεταφραστεί χρησιμοποιώντας την υπηρεσία μετάφρασης με τεχνητή νοημοσύνη [Co-op Translator](https://github.com/Azure/co-op-translator). Ενώ επιδιώκουμε την ακρίβεια, παρακαλούμε να έχετε υπόψη ότι οι αυτοματοποιημένες μεταφράσεις ενδέχεται να περιέχουν λάθη ή ανακρίβειες. Το πρωτότυπο έγγραφο στη μητρική του γλώσσα πρέπει να θεωρείται η αυθεντική πηγή. Για κρίσιμες πληροφορίες, συνιστάται επαγγελματική ανθρώπινη μετάφραση. Δεν φέρουμε ευθύνη για τυχόν παρεξηγήσεις ή λανθασμένες ερμηνείες που προκύπτουν από τη χρήση αυτής της μετάφρασης.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
